In [1]:
import os
import sys
import random
from pathlib import Path
from typing import Tuple, Dict, List

from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import numpy as np
import cv2

# ----------------------------
# Config
# ----------------------------
TARGET_PER_PROVINCE = 2000
TRAIN_FRACTION = 0.8
BANGKOK_NAME = "กรุงเทพมหานคร"

# Output into existing train split folder (expects: data/lower_train/data + labels.csv)
LOWER_TRAIN_REL = Path("data") / "lower_train"
LOWER_TRAIN_DATA_SUBDIR = "data"  # images live in data/lower_train/data/

# Province distribution (overall counts)
PROVINCE_DISTRIBUTION_REL = Path("debugging") / "exports" / "province_distribution.csv"

# Image generation
TARGET_W, TARGET_H = 128, 32
MARGIN_HORIZONTAL = 100  # total left+right margin (50px each side at 2x scale)
MARGIN_VERTICAL = 60     # total top+bottom margin (30px each side at 2x scale)

RANDOM_SEED = 42  # set None to make non-deterministic
if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)


def find_project_root(start: Path) -> Path:
    """Find repo root so notebook works from synthetic_data/ or repo root."""
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "train" / "province_mapping.py").exists() and (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import ISO province code mapping (TH-XX -> Thai name)
from train.province_mapping import PROVINCE_CODE_TO_THAI_NAME
THAI_NAME_TO_ISO: Dict[str, str] = {thai_name: iso for iso, thai_name in PROVINCE_CODE_TO_THAI_NAME.items()}

LOWER_TRAIN_DIR = PROJECT_ROOT / LOWER_TRAIN_REL
IMAGES_DIR = LOWER_TRAIN_DIR / LOWER_TRAIN_DATA_SUBDIR
LABELS_PATH = LOWER_TRAIN_DIR / "labels.csv"
DISTRIBUTION_CSV = PROJECT_ROOT / PROVINCE_DISTRIBUTION_REL

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOWER_TRAIN_DIR:", LOWER_TRAIN_DIR)
print("IMAGES_DIR:", IMAGES_DIR)
print("LABELS_PATH:", LABELS_PATH)
print("DISTRIBUTION_CSV:", DISTRIBUTION_CSV)

if not LOWER_TRAIN_DIR.exists():
    raise FileNotFoundError(f"Missing {LOWER_TRAIN_DIR} (expected existing lower_train)")
if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Missing {LABELS_PATH} (expected existing labels.csv)")
if not DISTRIBUTION_CSV.exists():
    raise FileNotFoundError(f"Missing {DISTRIBUTION_CSV} (expected province_distribution.csv)")

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Provinces list (ตาม full_provinces เดิม)
# - จะข้ามการเจน "กรุงเทพมหานคร" (มีพอแล้ว)
# ----------------------------
full_provinces = [
    "กรุงเทพมหานคร",
    "กระบี่",
    "กาญจนบุรี",
    "กาฬสินธุ์",
    "กำแพงเพชร",
    "ขอนแก่น",
    "จันทบุรี",
    "ฉะเชิงเทรา",
    "ชลบุรี",
    "ชัยนาท",
    "ชัยภูมิ",
    "ชุมพร",
    "เชียงราย",
    "เชียงใหม่",
    "ตรัง",
    "ตราด",
    "ตาก",
    "นครนายก",
    "นครปฐม",
    "นครพนม",
    "นครราชสีมา",
    "นครศรีธรรมราช",
    "นครสวรรค์",
    "นนทบุรี",
    "นราธิวาส",
    "น่าน",
    "บึงกาฬ",
    "บุรีรัมย์",
    "ปทุมธานี",
    "ประจวบคีรีขันธ์",
    "ปราจีนบุรี",
    "ปัตตานี",
    "พระนครศรีอยุธยา",
    "พังงา",
    "พัทลุง",
    "พิจิตร",
    "พิษณุโลก",
    "เพชรบุรี",
    "เพชรบูรณ์",
    "แพร่",
    "พะเยา",
    "ภูเก็ต",
    "มหาสารคาม",
    "มุกดาหาร",
    "แม่ฮ่องสอน",
    "ยโสธร",
    "ยะลา",
    "ร้อยเอ็ด",
    "ระนอง",
    "ระยอง",
    "ราชบุรี",
    "ลพบุรี",
    "ลำปาง",
    "ลำพูน",
    "เลย",
    "ศรีสะเกษ",
    "สกลนคร",
    "สงขลา",
    "สตูล",
    "สมุทรปราการ",
    "สมุทรสงคราม",
    "สมุทรสาคร",
    "สระแก้ว",
    "สระบุรี",
    "สิงห์บุรี",
    "สุโขทัย",
    "สุพรรณบุรี",
    "สุราษฎร์ธานี",
    "สุรินทร์",
    "หนองคาย",
    "หนองบัวลำภู",
    "อ่างทอง",
    "อำนาจเจริญ",
    "อุดรธานี",
    "อุตรดิตถ์",
    "อุทัยธานี",
    "อุบลราชธานี",
    "เบตง",
]

if BANGKOK_NAME not in full_provinces:
    raise ValueError(f"Bangkok name '{BANGKOK_NAME}' not found in full_provinces")

print(f"Total classes in full_provinces: {len(full_provinces)}")
print(f"Classes to generate (excluding Bangkok): {len([p for p in full_provinces if p != BANGKOK_NAME])}")

# ----------------------------
# Background palettes
# ----------------------------
palette = {
    "group1_personal": {
        "colors": [
            (255, 255, 255), (245, 245, 245), (230, 230, 230),
            (220, 225, 230), (210, 210, 210),
        ],
        "weight": 0.8,
    },
    "group2_taxi": {
        "colors": [
            (255, 235, 100), (255, 200, 90), (255, 180, 90),
            (150, 255, 150), (190, 240, 120),
        ],
        "weight": 0.1,
    },
    "group3_graphic": {
        "colors": [
            (255, 210, 230), (200, 220, 255), (255, 220, 170),
            (210, 200, 255), (180, 200, 210),
        ],
        "weight": 0.1,
    },
}


def pick_font() -> ImageFont.ImageFont:
    preferred = "C:/Windows/Fonts/Sarun's ThangLuang.ttf"
    candidates = [
        preferred,
        "C:/Windows/Fonts/tahoma.ttf",
        "C:/Windows/Fonts/THSarabunNew.ttf",
        "C:/Windows/Fonts/THSarabunNew Bold.ttf",
    ]
    for path in candidates:
        if os.path.exists(path):
            try:
                return ImageFont.truetype(path, 32)  # Larger font for 2x canvas
            except OSError:
                continue
    print("Warning: Thai font not found, using default")
    return ImageFont.load_default()


font = pick_font()


def choose_palette() -> Tuple[str, Tuple[int, int, int]]:
    groups = list(palette.keys())
    weights = [palette[g]["weight"] for g in groups]
    group = random.choices(groups, weights=weights, k=1)[0]
    color = random.choice(palette[group]["colors"])
    return group, color


def draw_province(province: str, bg_color: Tuple[int, int, int]) -> Image.Image:
    temp_img = Image.new("RGB", (1, 1))
    temp_draw = ImageDraw.Draw(temp_img)
    bbox = temp_draw.textbbox((0, 0), province, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    w = text_w + MARGIN_HORIZONTAL + random.randint(-5, 10)
    h = text_h + MARGIN_VERTICAL + random.randint(-3, 5)

    img = Image.new("RGB", (w, h), bg_color)
    draw = ImageDraw.Draw(img)
    draw.text((w // 2, h // 2), province, fill=(50, 50, 50), font=font, anchor="mm")
    return img


def apply_homography(img: np.ndarray) -> np.ndarray:
    height, width = img.shape[:2]
    src_points = np.float32([[0, 0], [width, 0], [width, height], [0, height]])
    distortion = random.uniform(0.03, 0.08)
    dst_points = np.float32(
        [
            [random.uniform(0, width * distortion), random.uniform(0, height * distortion)],
            [width - random.uniform(0, width * distortion), random.uniform(0, height * distortion)],
            [width - random.uniform(0, width * distortion), height - random.uniform(0, height * distortion)],
            [random.uniform(0, width * distortion), height - random.uniform(0, height * distortion)],
        ]
    )
    matrix = cv2.getPerspectiveTransform(src_points, dst_points)
    return cv2.warpPerspective(
        img,
        matrix,
        (width, height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE,
    )


def apply_augmentation(img: np.ndarray) -> np.ndarray:
    height, width = img.shape[:2]

    angle = random.uniform(-10, 10)
    rotation_matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    img = cv2.warpAffine(
        img,
        rotation_matrix,
        (width, height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE,
    )

    scale = random.uniform(0.85, 1.15)
    new_width = max(1, int(width * scale))
    new_height = max(1, int(height * scale))
    img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)

    if scale > 1.0:
        start_x = (new_width - width) // 2
        start_y = (new_height - height) // 2
        img = img[start_y : start_y + height, start_x : start_x + width]
    else:
        pad_x = (width - new_width) // 2
        pad_y = (height - new_height) // 2
        img = cv2.copyMakeBorder(
            img,
            pad_y,
            height - new_height - pad_y,
            pad_x,
            width - new_width - pad_x,
            cv2.BORDER_REPLICATE,
        )

    tx = random.randint(-20, 20)
    ty = random.randint(-10, 10)
    translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    img = cv2.warpAffine(img, translation_matrix, (width, height), borderMode=cv2.BORDER_REPLICATE)
    return img


def add_noise_and_blur(img: np.ndarray) -> np.ndarray:
    if random.random() < 0.3:
        if img.ndim == 3:
            noise = np.random.normal(0, 10, (img.shape[0], img.shape[1], 1)).astype(np.int16)
        else:
            noise = np.random.normal(0, 10, img.shape).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    if random.random() < 1.0:
        kernel_size = random.choice([3, 5])
        img = cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

    if random.random() < 0.3:
        brightness = random.uniform(0.8, 1.2)
        img = cv2.convertScaleAbs(img, alpha=brightness, beta=0)

    return img


def apply_bicubic_interpolation(img: np.ndarray, scale: float = 0.5) -> np.ndarray:
    height, width = img.shape[:2]
    small = cv2.resize(
        img,
        (max(1, int(width * scale)), max(1, int(height * scale))),
        interpolation=cv2.INTER_CUBIC,
    )
    return cv2.resize(small, (width, height), interpolation=cv2.INTER_CUBIC)


def thicken_text(image: np.ndarray) -> np.ndarray:
    k_size = 2
    kernel = np.ones((k_size, k_size), np.uint8)
    return cv2.erode(image, kernel, iterations=1)


def apply_heavy_motion_blur(image: np.ndarray) -> np.ndarray:
    kernel_size = random.choice([5, 7, 9, 11])
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[int((kernel_size - 1) / 2), :] = np.ones(kernel_size)
    kernel /= kernel_size
    return cv2.filter2D(image, -1, kernel)


def fade_image(image: np.ndarray) -> np.ndarray:
    h, w, c = image.shape
    gray_val = random.randint(100, 180)
    gray_overlay = np.full((h, w, c), gray_val, dtype=np.uint8)
    alpha = random.uniform(0.1, 0.25)
    return cv2.addWeighted(image, 1 - alpha, gray_overlay, alpha, 0)


def apply_squash(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    squash_factor = random.uniform(0.55, 0.80)
    new_h = max(1, int(h * squash_factor))
    return cv2.resize(img, (w, new_h), interpolation=cv2.INTER_AREA)


def load_distribution(path: Path) -> Dict[str, int]:
    df = pd.read_csv(path)
    if "province_description" not in df.columns or "count" not in df.columns:
        raise ValueError("province_distribution.csv must contain columns: province_description,count")
    df["province_description"] = df["province_description"].astype(str).str.strip()
    df["count"] = pd.to_numeric(df["count"], errors="coerce").fillna(0).astype(int)
    return dict(zip(df["province_description"], df["count"]))


dist = load_distribution(DISTRIBUTION_CSV)

# ----------------------------
# Compute quotas per province
# Rule: for each province, use count*0.8 (train split estimate) then generate to reach 2000.
# Skip Bangkok generation.
# ----------------------------
plan: List[Dict[str, object]] = []
for province in full_provinces:
    if province == BANGKOK_NAME:
        continue
    total_count = int(dist.get(province, 0))
    train_est = int(round(total_count * TRAIN_FRACTION))
    need = max(0, TARGET_PER_PROVINCE - train_est)
    iso_code = THAI_NAME_TO_ISO.get(province, "")  # empty for เบตง or unknown
    plan.append(
        {
            "province": province,
            "iso_code": iso_code,
            "count_total": total_count,
            "count_train_est": train_est,
            "target": TARGET_PER_PROVINCE,
            "need_to_generate": need,
        }
    )

plan_df = pd.DataFrame(plan)

print("\n--- Generation Plan (top 20 by need_to_generate) ---")
display(plan_df.sort_values("need_to_generate", ascending=False).head(20))
print(f"\nTotal images to generate (excluding Bangkok): {int(plan_df['need_to_generate'].sum()):,}")

# ----------------------------
# Append rows to existing labels.csv (streaming; avoids loading full CSV)
# ----------------------------
existing_cols = pd.read_csv(LABELS_PATH, nrows=0).columns.tolist()
if "filename" not in existing_cols:
    raise ValueError("labels.csv must contain a 'filename' column")


def append_rows_to_labels(rows: List[Dict[str, object]]):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    for col in existing_cols:
        if col not in df_new.columns:
            df_new[col] = ""
    df_new = df_new[existing_cols]
    df_new.to_csv(LABELS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")


def existing_synth_count_for_prefix(prefix: str) -> int:
    return len(list(IMAGES_DIR.glob(f"{prefix}*_lower.jpg")))

# ----------------------------
# Generate images province-by-province
# ----------------------------
generated_total = 0

for _, row in plan_df.iterrows():
    province = str(row["province"]).strip()
    iso_code = str(row["iso_code"]).strip()
    need = int(row["need_to_generate"])
    if need <= 0:
        continue

    prov_id = full_provinces.index(province)
    prefix = f"synth_prov{prov_id:03d}_"
    start_idx = existing_synth_count_for_prefix(prefix)

    buffer_rows: List[Dict[str, object]] = []

    for j in range(need):
        _, bg = choose_palette()
        img_pil = draw_province(province, bg)
        img_np = np.array(img_pil)

        # --- PHASE 0: Squash ---
        img_np = apply_squash(img_np)

        # --- PHASE 1: Geometry ---
        if random.random() < 0.5:
            img_np = apply_homography(img_np)
        if random.random() < 0.6:
            img_np = apply_augmentation(img_np)

        # --- PHASE 2: Structure ---
        if random.random() < 0.3:
            img_np = thicken_text(img_np)

        # --- PHASE 3: Motion ---
        if random.random() < 0.8:
            img_np = apply_heavy_motion_blur(img_np)
        elif random.random() < 0.3:
            img_np = add_noise_and_blur(img_np)

        # --- PHASE 4: Lighting/Sensor ---
        if random.random() < 0.6:
            img_np = fade_image(img_np)
        if random.random() < 0.25:
            img_np = add_noise_and_blur(img_np)

        # --- PHASE 5: Resolution ---
        if random.random() < 0.3:
            img_np = apply_bicubic_interpolation(img_np, scale=random.uniform(0.5, 0.8))

        # --- Final Resize ---
        img_np = cv2.resize(img_np, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

        fname = f"{prefix}{start_idx + j:06d}_lower.jpg"
        out_path = IMAGES_DIR / fname
        if out_path.exists():
            k = start_idx + j
            while out_path.exists():
                k += 1
                fname = f"{prefix}{k:06d}_lower.jpg"
                out_path = IMAGES_DIR / fname

        cv2.imwrite(str(out_path), cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))

        buffer_rows.append(
            {
                "filename": fname,
                "label": province,
                "source": "synthetic",
                "transactionDate": "",
                "plate": "",
                "province_code": iso_code,
                "province_description": province,
                "brand_description": "",
                "colors_code": "",
                "colors_description": "",
                "vehicleClass": "1.0",
            }
        )

        generated_total += 1

        # flush every 2000 rows (per province target)
        if len(buffer_rows) >= 2000:
            append_rows_to_labels(buffer_rows)
            buffer_rows = []

    append_rows_to_labels(buffer_rows)
    print(f"Generated {need:,} for {province} (prov_id={prov_id}, iso={iso_code or 'N/A'})")

print(f"\nDone. Generated total: {generated_total:,}")
print(f"Images written to: {IMAGES_DIR}")
print(f"Labels appended to: {LABELS_PATH}")


PROJECT_ROOT: C:\Users\Tanaphat\Desktop\Coding\ALPR
LOWER_TRAIN_DIR: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\lower_train
IMAGES_DIR: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\lower_train\data
LABELS_PATH: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\lower_train\labels.csv
DISTRIBUTION_CSV: C:\Users\Tanaphat\Desktop\Coding\ALPR\debugging\exports\province_distribution.csv
Total classes in full_provinces: 78
Classes to generate (excluding Bangkok): 77

--- Generation Plan (top 20 by need_to_generate) ---


,province,iso_code,count_total,count_train_est,target,need_to_generate
76,เบตง,,0,0,2000,2000
23,นราธิวาส,TH-96,0,0,2000,2000
52,ลำพูน,TH-51,0,0,2000,2000
47,ระนอง,TH-85,0,0,2000,2000
45,ยะลา,TH-95,0,0,2000,2000
39,พะเยา,TH-56,0,0,2000,2000
30,ปัตตานี,TH-94,0,0,2000,2000
69,หนองบัวลำภู,TH-39,1,1,2000,1999
59,สมุทรสงคราม,TH-75,1,1,2000,1999
57,สตูล,TH-91,1,1,2000,1999



Total images to generate (excluding Bangkok): 152,871
Generated 1,999 for กระบี่ (prov_id=1, iso=TH-81)
Generated 1,998 for กาญจนบุรี (prov_id=2, iso=TH-71)
Generated 1,993 for กาฬสินธุ์ (prov_id=3, iso=TH-46)
Generated 1,986 for กำแพงเพชร (prov_id=4, iso=TH-62)
Generated 1,978 for ขอนแก่น (prov_id=5, iso=TH-40)
Generated 1,992 for จันทบุรี (prov_id=6, iso=TH-22)
Generated 1,978 for ฉะเชิงเทรา (prov_id=7, iso=TH-24)
Generated 1,821 for ชลบุรี (prov_id=8, iso=TH-20)
Generated 1,990 for ชัยนาท (prov_id=9, iso=TH-18)
Generated 1,991 for ชัยภูมิ (prov_id=10, iso=TH-36)
Generated 1,997 for ชุมพร (prov_id=11, iso=TH-86)
Generated 1,990 for เชียงราย (prov_id=12, iso=TH-57)
Generated 1,970 for เชียงใหม่ (prov_id=13, iso=TH-50)
Generated 1,997 for ตรัง (prov_id=14, iso=TH-92)
Generated 1,998 for ตราด (prov_id=15, iso=TH-23)
Generated 1,995 for ตาก (prov_id=16, iso=TH-63)
Generated 1,985 for นครนายก (prov_id=17, iso=TH-26)
Generated 1,990 for นครปฐม (prov_id=18, iso=TH-73)
Generated 1,998 for น

In [2]:
# Cell 2: Generate LOWER TEST synthetic
# - Generate 200 per province (78 provinces; including Bangkok + Betong)
# - Save images to: data/lower_test_synthetic/data/
# - Save labels to: data/lower_test_synthetic/labels.csv

from pathlib import Path
import random
import pandas as pd
import numpy as np
import cv2

# ---- Settings ----
NUM_PER_PROVINCE = 200
LOWER_TEST_SYNTH_REL = Path("data") / "lower_test_synthetic"
LOWER_TEST_SYNTH_DATA_SUBDIR = "data"

# ---- Reuse mapping/list from Cell 1 if available ----
try:
    PROJECT_ROOT  # noqa: B018
    full_provinces  # noqa: B018
    THAI_NAME_TO_ISO  # noqa: B018
except NameError as e:
    raise RuntimeError(
        "Cell 2 expects Cell 1 to be run first (for province list + font + augmentation functions). "
        "Run Cell 1, then run Cell 2."
    ) from e

TEST_DIR = PROJECT_ROOT / LOWER_TEST_SYNTH_REL
TEST_IMAGES_DIR = TEST_DIR / LOWER_TEST_SYNTH_DATA_SUBDIR
TEST_LABELS_PATH = TEST_DIR / "labels.csv"

TEST_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Use same CSV schema as training labels (for compatibility)
# If you want a custom schema, change this list.
CSV_COLUMNS = [
    "filename",
    "label",
    "source",
    "transactionDate",
    "plate",
    "province_code",
    "province_description",
    "brand_description",
    "colors_code",
    "colors_description",
    "vehicleClass",
]


def existing_synth_count_for_prefix_in_dir(images_dir: Path, prefix: str) -> int:
    return len(list(images_dir.glob(f"{prefix}*_lower.jpg")))


def write_labels_csv_header_if_missing(path: Path):
    if not path.exists():
        pd.DataFrame([], columns=CSV_COLUMNS).to_csv(path, index=False, encoding="utf-8-sig")


def append_rows_to_test_labels(rows):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    for col in CSV_COLUMNS:
        if col not in df_new.columns:
            df_new[col] = ""
    df_new = df_new[CSV_COLUMNS]
    df_new.to_csv(TEST_LABELS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")


write_labels_csv_header_if_missing(TEST_LABELS_PATH)

print("TEST_DIR:", TEST_DIR)
print("TEST_IMAGES_DIR:", TEST_IMAGES_DIR)
print("TEST_LABELS_PATH:", TEST_LABELS_PATH)
print("NUM_PER_PROVINCE:", NUM_PER_PROVINCE)
print("Total provinces:", len(full_provinces))

# Make the generation deterministic-ish but different stream than Cell 1
# (keeps results stable even if Cell 1 used RANDOM_SEED)
random.seed(2026)
np.random.seed(2026)

# Generate exactly NUM_PER_PROVINCE for every province in full_provinces (including Bangkok)
generated_total = 0

for province in full_provinces:
    prov_id = full_provinces.index(province)
    iso_code = THAI_NAME_TO_ISO.get(province, "")  # Betong will be empty

    prefix = f"synth_test_prov{prov_id:03d}_"
    start_idx = existing_synth_count_for_prefix_in_dir(TEST_IMAGES_DIR, prefix)

    buffer_rows = []

    for j in range(NUM_PER_PROVINCE):
        _, bg = choose_palette()
        img_pil = draw_province(province, bg)
        img_np = np.array(img_pil)

        # Same augmentation pipeline as Cell 1 (keeps style consistent)
        img_np = apply_squash(img_np)

        if random.random() < 0.5:
            img_np = apply_homography(img_np)
        if random.random() < 0.6:
            img_np = apply_augmentation(img_np)

        if random.random() < 0.3:
            img_np = thicken_text(img_np)

        if random.random() < 0.8:
            img_np = apply_heavy_motion_blur(img_np)
        elif random.random() < 0.3:
            img_np = add_noise_and_blur(img_np)

        if random.random() < 0.6:
            img_np = fade_image(img_np)
        if random.random() < 0.25:
            img_np = add_noise_and_blur(img_np)

        if random.random() < 0.3:
            img_np = apply_bicubic_interpolation(img_np, scale=random.uniform(0.5, 0.8))

        img_np = cv2.resize(img_np, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

        fname = f"{prefix}{start_idx + j:06d}_lower.jpg"
        out_path = TEST_IMAGES_DIR / fname
        if out_path.exists():
            k = start_idx + j
            while out_path.exists():
                k += 1
                fname = f"{prefix}{k:06d}_lower.jpg"
                out_path = TEST_IMAGES_DIR / fname

        cv2.imwrite(str(out_path), cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))

        buffer_rows.append(
            {
                "filename": fname,
                "label": province,
                "source": "synthetic_test",
                "transactionDate": "",
                "plate": "",
                "province_code": iso_code,
                "province_description": province,
                "brand_description": "",
                "colors_code": "",
                "colors_description": "",
                "vehicleClass": "1.0",
            }
        )

        generated_total += 1

    append_rows_to_test_labels(buffer_rows)
    print(f"Generated {NUM_PER_PROVINCE} for {province} (prov_id={prov_id}, iso={iso_code or 'N/A'})")

print(f"\nDone. Generated total: {generated_total:,} ({NUM_PER_PROVINCE} x {len(full_provinces)})")
print(f"Images written to: {TEST_IMAGES_DIR}")
print(f"Labels written to: {TEST_LABELS_PATH}")


TEST_DIR: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\lower_test_synthetic
TEST_IMAGES_DIR: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\lower_test_synthetic\data
TEST_LABELS_PATH: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\lower_test_synthetic\labels.csv
NUM_PER_PROVINCE: 200
Total provinces: 78
Generated 200 for กรุงเทพมหานคร (prov_id=0, iso=TH-10)
Generated 200 for กระบี่ (prov_id=1, iso=TH-81)
Generated 200 for กาญจนบุรี (prov_id=2, iso=TH-71)
Generated 200 for กาฬสินธุ์ (prov_id=3, iso=TH-46)
Generated 200 for กำแพงเพชร (prov_id=4, iso=TH-62)
Generated 200 for ขอนแก่น (prov_id=5, iso=TH-40)
Generated 200 for จันทบุรี (prov_id=6, iso=TH-22)
Generated 200 for ฉะเชิงเทรา (prov_id=7, iso=TH-24)
Generated 200 for ชลบุรี (prov_id=8, iso=TH-20)
Generated 200 for ชัยนาท (prov_id=9, iso=TH-18)
Generated 200 for ชัยภูมิ (prov_id=10, iso=TH-36)
Generated 200 for ชุมพร (prov_id=11, iso=TH-86)
Generated 200 for เชียงราย (prov_id=12, iso=TH-57)
Generated 200 for เชียงใหม่ (prov_id=13, iso=TH-50)
G